# Домашнее задание 4
## Околокруговое приближение. Одноимпульсные манёвры. Базис-вектор.

**Введение в маневрирование КА · К лекции 5**  
**Срок сдачи: 26 октября 2026 года, 23:59.**

Решите задачу перехода между близкими компланарными орбитами
в околокруговом приближении.

**Задания 1–3 обязательны; задание 4* — по желанию.**


### Как работать с ноутбуком

Выполняйте ячейки по порядку. Нужны Python 3, NumPy и Matplotlib.
Готовые функции выводят таблицы, строят график и позволяют при желании проверить ответы;
их код менять не требуется. Самостоятельно получите формулы, заполните
переменные со значением `None` и текстовые ячейки с ответами.

**Чекеры предназначены только для самопроверки. Использовать их не обязательно.**
Ячейки с их вызовами можно пропустить; сохранять результаты проверок не нужно.

**Единицы и порядок компонент:**
- `phi_2`, `phi_3`: одномерные массивы углов в **радианах**, в диапазоне $[0,2\pi)$;
- `dv_hat_2`, `dv_hat_3`: массивы формы `(2, 2)`, каждая строка —
  $(\Delta\widehat V_r,\Delta\widehat V_t)$; компоненты **делятся на $V_0$**;
- в задании 2 строки — **два отдельных одноимпульсных решения**;
  в задании 3 — **два импульса одного перехода**;
- `lambdas_3`: три числа в порядке $(\lambda_1,\lambda_2,\lambda_3)$;
  остальные множители равны нулю;
- в таблицах углы автоматически переводятся в градусы, импульсы — в м/с.

Не округляйте промежуточные результаты. Успешные численные проверки
не заменяют выводы, доказательства и объяснения. Пока ответы не заполнены,
ячейки выводят напоминания и не прерывают выполнение ноутбука.


## Задание 1. Чтение методички

В методичке прочитайте разделы **1.5, 1.9 и 2.2–2.4**: импульсное изменение параметров,
постановку задачи перехода, базис-вектор и условия оптимальности.
Раздел 2.1 используйте для справки по обозначениям.

1. Почему выполнение граничных условий само по себе не доказывает
   оптимальность манёвра?
2. Какие условия на манёвр и один общий базис-вектор достаточны для
   глобального минимума характеристической скорости? Почему проверки модуля только в точках импульсов
   недостаточно?


**Ответ на вопрос 1:**

*Впишите ответ и обоснование.*

**Ответ на вопрос 2:**

*Перечислите условия и объясните, при каких углах их необходимо проверять.*


## Общие исходные данные

$$\mu=398600{,}44158\ \text{км}^3/\text{с}^2,\qquad
  r_0=a_i=7000\ \text{км},\qquad V_0=\sqrt{\mu/r_0}.$$
$$e_{xi}=e_{yi}=0,\qquad a_f=7007\ \text{км},\qquad
  e_{xf}=0{,}0012,\qquad e_{yf}=0{,}0016.$$

Ось отсчёта углов фиксирована в общей плоскости орбит;
$e_x,e_y$ обеих орбит заданы относительно этой оси.

$$\Delta a=\frac{a_f-a_i}{r_0},\quad
  \Delta e_x=e_{xf}-e_{xi},\quad \Delta e_y=e_{yf}-e_{yi},\quad
  \Delta e=\sqrt{(\Delta e_x)^2+(\Delta e_y)^2}.$$

Нормальная компонента импульса равна нулю.
Крышка означает деление компоненты импульса на $V_0$.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager

ACCENT = "#187AA5"
SECOND = "#A36540"
INK = "#302B28"
fonts = {font.name for font in font_manager.fontManager.ttflist}
plt.rcParams.update({
    "font.family": "PT Serif" if "PT Serif" in fonts else "DejaVu Serif",
    "font.size": 11,
    "text.color": INK,
    "axes.labelcolor": INK,
    "axes.titleweight": "normal",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "figure.dpi": 110,
})


In [ ]:
MU = 398600.44158       # км³/с²
R0 = 7000.0            # км
V0 = np.sqrt(MU / R0)  # км/с
AI = R0
AF = 7007.0           # км; основной случай заданий 2 и 3
EXI, EYI = 0.0, 0.0
EXF, EYF = 0.0012, 0.0016

DA = (AF - AI) / R0
DEX, DEY = EXF - EXI, EYF - EYI
DE = np.hypot(DEX, DEY)
TWO_PI = 2 * np.pi

print(f"V₀ = {V0:.8f} км/с")
print(f"Δa = {DA:.8f}; Δeₓ = {DEX:.8f}; Δeᵧ = {DEY:.8f}; Δe = {DE:.8f}")


### Готовые функции для таблиц, графика и самопроверки

Выполните следующие ячейки. Разбирать их код для выполнения задания не нужно.

`impulse_table(phi, dv_hat, title, together=False)` печатает углы,
компоненты и модули. При `together=True` также выводит сумму модулей.

`primer(phi, lambdas)` возвращает две компоненты $(s_r,s_t)$ базис-вектора.
`plot_primer(phi_impulses, lambdas)` строит его модуль за оборот.

`check_task2(...)`, `check_task3(...)`, `check_star(...)` возвращают `True`
при успешной численной проверке и `False`, если обнаружена ошибка.
Если ответы ещё не заполнены, возвращается `None`.
Сообщения поясняют, какое условие стоит проверить.


In [ ]:
def orbit_changes(phi, dv_hat):
    """Изменения (Δa, Δeₓ, Δeᵧ) от каждой строки импульсов, околокруговое приближение."""
    phi = np.asarray(phi, dtype=float)
    dv = np.asarray(dv_hat, dtype=float)
    vr, vt = dv[:, 0], dv[:, 1]
    return np.column_stack((2 * vt,
                            vr * np.sin(phi) + 2 * vt * np.cos(phi),
                            -vr * np.cos(phi) + 2 * vt * np.sin(phi)))


def primer(phi, lambdas):
    """Плоский базис-вектор при λ₄ = λ₅ = λ₆ = 0; последний индекс: r,t."""
    phi = np.asarray(phi, dtype=float)
    l1, l2, l3 = np.asarray(lambdas, dtype=float)
    return np.stack((l3 * np.sin(phi) - l2 * np.cos(phi),
                     2 * l1 + 2 * l3 * np.cos(phi) + 2 * l2 * np.sin(phi)),
                    axis=-1)


def impulse_table(phi, dv_hat, title, together=False):
    phi = np.asarray(phi, dtype=float)
    dimensional = np.asarray(dv_hat, dtype=float) * V0 * 1000
    norms = np.linalg.norm(dimensional, axis=1)
    print(title)
    print(f"{'№':>3} {'φ, град':>13} {'ΔVᵣ, м/с':>15} {'ΔVₜ, м/с':>15} {'|ΔV|, м/с':>15}")
    for j, (angle, components, norm) in enumerate(zip(phi, dimensional, norms), 1):
        print(f"{j:3d} {np.degrees(angle):13.6f} {components[0]:15.6f} "
              f"{components[1]:15.6f} {norm:15.6f}")
    if together:
        print(f"Суммарная характеристическая скорость: {norms.sum():.6f} м/с")


def plot_primer(phi_impulses, lambdas):
    phi = np.linspace(0.0, TWO_PI, 1201)
    norm = np.linalg.norm(primer(phi, lambdas), axis=-1)
    impulse_norms = np.linalg.norm(primer(phi_impulses, lambdas), axis=-1)
    fig, ax = plt.subplots(figsize=(10, 4), constrained_layout=True)
    ax.plot(np.degrees(phi), norm, color=ACCENT, linewidth=2,
            label=r"$\|\boldsymbol{s}(\varphi)\|$")
    ax.axhline(1.0, color=SECOND, linestyle="--", label="Единичный модуль")
    ax.scatter(np.degrees(phi_impulses), impulse_norms, color=SECOND,
               zorder=4, label="Углы приложения импульсов")
    for j, (angle, value) in enumerate(zip(phi_impulses, impulse_norms), 1):
        ax.annotate(str(j), (np.degrees(angle), value), xytext=(5, 7),
                    textcoords="offset points", color=INK)
    ax.set(xlabel=r"Угол $\varphi$, град", ylabel="Модуль базис-вектора",
           title="Задание 3. Проверка модуля базис-вектора", xlim=(0, 360))
    ax.set_ylim(0, max(1.1, float(norm.max()) * 1.1))
    ax.set_xticks(np.arange(0, 361, 60))
    ax.grid(alpha=0.22)
    ax.legend(loc="best")
    return fig, ax


In [ ]:
# Простые численные проверки
_BOUNDARY_ATOL = 1e-8


def _report(title, checks):
    print(title)
    for passed, message in checks:
        print(("[OK] " if passed else "[Проверьте] ") + message)
    passed = all(bool(item[0]) for item in checks)
    print("Численные проверки пройдены." if passed
          else "Есть замечания; исправьте отмеченные пункты и повторите проверку.")
    return passed


def _maneuver_arrays(phi, dv_hat):
    try:
        phi = np.asarray(phi, dtype=float)
        dv = np.asarray(dv_hat, dtype=float)
    except (ValueError, TypeError):
        print("Углы и компоненты должны быть числовыми массивами без None.")
        return None
    if phi.shape != (2,) or dv.shape != (2, 2):
        print("Нужны два угла и массив формы (2, 2): строки — импульсы, столбцы — r,t.")
        return None
    if not np.all(np.isfinite(phi)) or not np.all(np.isfinite(dv)):
        print("В ответе есть NaN или бесконечность. Проверьте вычисления.")
        return None
    if np.any(phi < 0) or np.any(phi >= TWO_PI):
        print("Углы задавайте в радианах в [0, 2π). Проверьте перевод из градусов.")
        return None
    return phi, dv


def check_task2(phi, dv_hat, counts):
    """counts — словарь {7014: число решений, 7021: число решений}."""
    if phi is None or dv_hat is None:
        print("Задание 2: сначала заполните phi_2 и dv_hat_2 своими ответами.")
        return None
    data = _maneuver_arrays(phi, dv_hat)
    if data is None:
        return False
    phi, dv = data
    changes = orbit_changes(phi, dv)
    target = np.array([DA, DEX, DEY])
    distance = abs(np.angle(np.exp(1j * (phi[0] - phi[1]))))
    checks = [(distance > 1e-6, "Найдены две различные точки, без дублирования углов.")]
    for k, name in enumerate(("Δa", "Δeₓ", "Δeᵧ")):
        error = float(np.max(np.abs(changes[:, k] - target[k])))
        checks.append((error <= _BOUNDARY_ATOL,
                       f"Условие на {name} для КАЖДОГО решения; макс. невязка {error:.2e}."))
    norms = np.linalg.norm(dv, axis=1)
    checks.append((abs(norms[0] - norms[1]) <= _BOUNDARY_ATOL,
                   "Модули двух одноимпульсных решений совпадают."))
    for af, expected in ((7014, 1), (7021, 0)):
        answer = counts.get(af) if isinstance(counts, dict) else None
        valid_integer = isinstance(answer, (int, np.integer)) and not isinstance(answer, (bool, np.bool_))
        checks.append((valid_integer and answer == expected,
                       f"Число решений при a_f = {af} км; проверьте условие существования и касание."))
    return _report("Самопроверка задания 2", checks)


def check_task3(phi, dv_hat, lambdas):
    """Проверяет одну двухимпульсную схему и общий для неё базис-вектор."""
    if phi is None or dv_hat is None or lambdas is None:
        print("Задание 3: сначала заполните phi_3, dv_hat_3 и lambdas_3.")
        return None
    data = _maneuver_arrays(phi, dv_hat)
    if data is None:
        return False
    phi, dv = data
    try:
        l = np.asarray(lambdas, dtype=float)
    except (TypeError, ValueError):
        print("Множители должны быть тремя числами: λ₁, λ₂, λ₃.")
        return False
    if l.shape != (3,) or not np.all(np.isfinite(l)):
        print("Нужен массив из трёх конечных чисел: [λ₁, λ₂, λ₃].")
        return False
    first = np.arctan2(DEY, DEX) % TWO_PI
    specified = np.array([first, (first + np.pi) % TWO_PI])
    distances = np.abs(np.angle(np.exp(1j * (phi[:, None] - specified[None, :]))))
    angles_ok = np.all(distances.min(axis=0) < 2e-5) and np.all(distances.min(axis=1) < 2e-5)
    total_change = orbit_changes(phi, dv).sum(axis=0)
    norm = np.linalg.norm(dv, axis=1)
    s = primer(phi, l)
    nonzero = bool(np.all(norm > 1e-12))
    directions_ok = nonzero and np.allclose(s, dv / norm[:, None], rtol=0, atol=2e-5)
    # Точный максимум модуля для этого плоского семейства на полном обороте.
    # В решении самостоятельно обоснуйте ограничение для найденных множителей.
    max_norm = 2 * (abs(l[0]) + np.hypot(l[1], l[2]))
    checks = [
        (angles_ok, "Углы соответствуют направлению Δe и противоположной точке орбиты."),
        (np.all(np.abs(dv[:, 0]) <= _BOUNDARY_ATOL), "Оба импульса чисто трансверсальные."),
        (nonzero, "Оба импульса ненулевые; направление каждого определено."),
    ]
    for k, name in enumerate(("Δa", "Δeₓ", "Δeᵧ")):
        error = abs(total_change[k] - (DA, DEX, DEY)[k])
        checks.append((error <= _BOUNDARY_ATOL,
                       f"Суммарное условие на {name}; невязка {error:.2e}."))
    checks.extend([
        (abs(l[0]) <= 1e-8, "В заданном семействе λ₁ = 0."),
        (directions_ok, "Один базис-вектор совпадает с направлениями ОБОИХ импульсов; учтены знаки."),
        (np.allclose(np.linalg.norm(s, axis=1), 1, rtol=0, atol=2e-5),
         "В обеих точках импульсов модуль базис-вектора равен единице."),
        (max_norm <= 1 + 1e-6,
         f"Ограничение модуля на всём обороте; максимальный модуль {max_norm:.8f}."),
    ])
    return _report("Самопроверка задания 3", checks)


In [ ]:
def check_star(min_speed_function, optimal_one_function):
    """Проверяет две скалярные функции задания 4* на контрольных примерах."""
    if not callable(min_speed_function) or not callable(optimal_one_function):
        print("Задание 4*: определите обе функции и передайте их в check_star.")
        return None
    # da, dex, dey; коэффициент при V₀; оптимален ли ненулевой один импульс.
    cases = [
        ((0.0, 0.0, 0.0), 0.0, False),
        ((0.001, 0.0, 0.0), 0.0005, False),
        ((-0.003, 0.0, 0.0), 0.0015, False),
        ((0.0, 0.0012, 0.0016), 0.001, False),
        ((0.001, 0.0012, 0.0016), 0.001, False),
        ((-0.001, 0.0012, 0.0016), 0.001, False),
        ((0.002, 0.0012, 0.0016), 0.001, True),
        ((-0.002, 0.0012, 0.0016), 0.001, True),
        ((0.004, 0.0012, 0.0016), 0.002, False),
        ((-0.004, 0.0012, 0.0016), 0.002, False),
        ((0.001, -0.0006, 0.0008), 0.0005, True),
        ((-0.001, 0.0008, -0.0006), 0.0005, True),
        ((0.0005, -0.0016, 0.0012), 0.001, False),
        ((0.003, 0.001, 0.0), 0.0015, False),
        ((0.0, -0.002, 0.0), 0.001, False),
        ((0.0, 0.0, -0.003), 0.0015, False),
    ]
    checks = []
    for values, speed_factor, reference_optimal in cases:
        label = f"(Δa, Δeₓ, Δeᵧ) = {values}"
        try:
            value = min_speed_function(*values)
            optimal = optimal_one_function(*values)
            speed = np.asarray(value, dtype=float)
            reference_speed = speed_factor * V0 * 1000  # м/с
            speed_ok = (value is not None and speed.shape == () and np.isfinite(speed)
                        and np.isclose(float(speed), reference_speed, rtol=1e-6, atol=1e-9))
            flag_ok = isinstance(optimal, (bool, np.bool_)) and bool(optimal) == reference_optimal
            checks.append((bool(speed_ok), "Оптимальное значение J, м/с: " + label))
            checks.append((bool(flag_ok), "Критерий оптимальности ненулевого одного импульса: " + label))
        except Exception as error:
            checks.append((False, f"Ошибка для {label}: {type(error).__name__}: {error}"))
    return _report("Самопроверка задания 4* (контрольные примеры, не доказательство)", checks)


## Задание 2. Одноимпульсный переход

Используйте три граничных условия первого приближения:
$$\begin{aligned}
2\Delta\widehat V_t&=\Delta a,\\
\Delta\widehat V_r\sin\varphi+2\Delta\widehat V_t\cos\varphi&=\Delta e_x,\\
-\Delta\widehat V_r\cos\varphi+2\Delta\widehat V_t\sin\varphi&=\Delta e_y.
\end{aligned}$$

1. Исключите компоненты импульса и получите уравнение для угла $\varphi$.
   Выведите условие существования решения и определите случаи двух,
   одной и ни одной точки перехода. Случай $\Delta a=\Delta e=0$ здесь исключён.
2. Найдите все решения при $a_f=7007$ км: углы, компоненты и модули импульсов.
   Объясните совпадение модулей и связь с пересечением орбит.
3. Сохранив $e_{xf},e_{yf}$, определите число решений при $a_f=7014$ км и
   $a_f=7021$ км. Полный расчёт импульсов для этих случаев повторять не нужно.

> **Подсказка.** Умножьте вторую строку на $\cos\varphi$, третью на
> $\sin\varphi$ и сложите. Для радиальной компоненты используйте другую
> линейную комбинацию. Углы найдите аналитически из уравнений околокругового приближения.


**Уравнение для угла, условие существования и восстановление компонент:**

*Выведите формулы. Отдельно укажите случаи двух, одной и нуля точек перехода.*


In [ ]:
# Заполните своими результатами. Не подставляйте сюда компоненты в м/с!
# Например, используйте np.array([...]); значения вычислите по своим формулам.
phi_2 = None       # np.array([угол первого решения, угол второго решения]), рад
dv_hat_2 = None    # np.array([[Vr_hat_1, Vt_hat_1], [Vr_hat_2, Vt_hat_2]])
counts_2 = None    # словарь {7014: число решений, 7021: число решений}


In [ ]:
if phi_2 is not None and dv_hat_2 is not None:
    impulse_table(phi_2, dv_hat_2, "Два отдельных одноимпульсных решения")
else:
    print("Сначала заполните phi_2 и dv_hat_2.")


In [ ]:
check_task2(phi_2, dv_hat_2, counts_2)


**Объяснение результатов:**

*Почему модули совпали? Как связаны углы с пересечением орбит?
Как классифицируются случаи $a_f=7014$ км и $a_f=7021$ км в околокруговом приближении?*


## Задание 3. Два импульса и проверка оптимальности

Вернитесь к $a_f=7007$ км. Два импульса — чисто трансверсальные, приложенные при углах
$$\cos\varphi_1=\frac{\Delta e_x}{\Delta e},\quad
  \sin\varphi_1=\frac{\Delta e_y}{\Delta e},\quad
  \varphi_2=(\varphi_1+\pi)\pmod{2\pi}.$$

1. Найдите обе трансверсальные компоненты из граничных условий, сохраняя
   их знаки. Проверьте суммарные условия на $\Delta a,\Delta e_x,\Delta e_y$.
2. Вычислите сумму **модулей** импульсов в м/с и сравните с заданием 2.
3. Положите $\lambda_1=\lambda_4=\lambda_5=\lambda_6=0$.
   Найдите $\lambda_2,\lambda_3$ из совпадения
   $\boldsymbol s(\varphi_1)$ с единичным вектором первого импульса.
   Проверьте второй импульс с учётом знака его компоненты.
4. Постройте график модуля базис-вектора за оборот. Аналитически докажите
   $\|\boldsymbol s(\varphi)\|\leq1$ для всех углов и обоснуйте глобальный
   минимум среди манёвров с произвольным числом импульсов.

Используйте
$$s_r=\lambda_3\sin\varphi-\lambda_2\cos\varphi,\qquad
  s_t=2\lambda_1+2\lambda_3\cos\varphi+2\lambda_2\sin\varphi,\qquad s_z=0.$$

> **Подсказка.** Для найденных множителей выразите компоненты через
> $\varphi-\varphi_1$ и вычислите квадрат модуля. Одного графика для
> доказательства ограничения между узлами сетки недостаточно.


**Вывод компонент двух импульсов и проверка граничных условий:**

*Приведите формулы, подстановку в три суммарных условия и выражение для характеристической скорости.*


In [ ]:
phi_3 = None       # два заданных угла, рад; используйте np.arctan2 с учётом четверти
dv_hat_3 = None    # np.array([[0.0, Vt_hat_1], [0.0, Vt_hat_2]])


In [ ]:
if phi_3 is not None and dv_hat_3 is not None:
    impulse_table(phi_3, dv_hat_3, "Два импульса одного перехода", together=True)
    if dv_hat_2 is not None:
        one_speed = np.linalg.norm(np.asarray(dv_hat_2)[0]) * V0 * 1000
        two_speed = np.linalg.norm(np.asarray(dv_hat_3), axis=1).sum() * V0 * 1000
        print(f"Одноимпульсный переход: {one_speed:.6f} м/с")
        print(f"Разность характеристических скоростей: {one_speed - two_speed:.6f} м/с")
else:
    print("Сначала заполните phi_3 и dv_hat_3.")


**Определение множителей:**

*Запишите условия на $s_r(\varphi_1),s_t(\varphi_1)$, найдите множители
и проверьте направление в точке второго импульса.*


In [ ]:
lambdas_3 = None   # np.array([lambda_1, lambda_2, lambda_3]); lambda_1 = 0


In [ ]:
if phi_3 is not None and lambdas_3 is not None:
    fig, ax = plot_primer(phi_3, lambdas_3)
    plt.show()
else:
    print("Сначала заполните phi_3 и lambdas_3.")


In [ ]:
check_task3(phi_3, dv_hat_3, lambdas_3)


**Аналитическое доказательство и вывод:**

*Вычислите квадрат модуля и докажите ограничение на всём обороте.
Перечислите выполненные достаточные условия. Объясните, почему доказан
минимум среди манёвров с любым числом импульсов, а не только с двумя.*


## Задание 4*. Когда один импульс оптимален?

**Необязательное задание.** Рассмотрите произвольные малые разности
$\Delta a,\Delta e_x,\Delta e_y$ для компланарных орбит в той же постановке.
Решите задачу перехода в околокруговом приближении.
Найдите оптимальное значение суммарной характеристической скорости.
$$J_{\min}(\Delta a,\Delta e_x,\Delta e_y),\qquad
  J=V_0\sum_j\sqrt{(\Delta\widehat V_{rj})^2+(\Delta\widehat V_{tj})^2}.$$

1. Обобщите двухимпульсную схему задания 3 на произвольный $\Delta a$.
   При $\Delta e>0$ используйте направление $(\Delta e_x,\Delta e_y)$ и
   противоположную точку орбиты. Отдельно рассмотрите $\Delta e=0$ и нулевые разности.
2. Получите общую формулу для $J_{\min}$ и докажите её через базис-вектор.
   Разберите разные соотношения $|\Delta a|$ и $\Delta e$;
   при необходимости выбирайте разные множители.
3. Определите, когда существует **ненулевой одноимпульсный переход**,
   достигающий этого минимума. Отличайте невозможность перехода одним
   импульсом от существования такого перехода с большей характеристической скоростью.
4. Для самопроверки при желании реализуйте две функции ниже.
   Их можно проверить с помощью `check_star`.

**Соглашение для нулевого случая:** при полностью нулевых разностях
$J_{\min}=0$ достигается без манёвров. Это не ненулевой одноимпульсный переход;
функция `one_impulse_optimal_star` должна вернуть `False`.

> **Подсказка.** Допустимая схема даёт верхнюю оценку. Найдите базис-вектор,
> дающий ту же нижнюю оценку. Не забывайте модули отдельных импульсов;
> направление вектора эксцентриситета при $\Delta e=0$ не определено.


**Обобщение схемы, формула минимума и доказательство:**

*Приведите решение для всех случаев, включая вырожденные.
Запишите критерий оптимальности ненулевого одного импульса.*


In [ ]:
def j_min_star(da, dex, dey):
    """Верните оптимальное значение J в м/с."""
    # Ваше выражение для произвольных скалярных da, dex, dey.
    return None


def one_impulse_optimal_star(da, dex, dey):
    """Верните bool: достигается ли минимум характеристической скорости ненулевым одним импульсом."""
    # Ваш критерий. Учтите погрешность сравнения чисел и нулевой случай.
    return None


In [ ]:
# Пока функции возвращают None, выводится напоминание.
if (j_min_star(DA, DEX, DEY) is None
        or one_impulse_optimal_star(DA, DEX, DEY) is None):
    print("Задание 4* необязательное. Для самопроверки сначала заполните обе функции.")
else:
    check_star(j_min_star, one_impulse_optimal_star)


Чекер задания 4* проверяет численные ответы на контрольных наборах с разными
знаками $\Delta a$, направлениями вектора эксцентриситета, граничными и нулевыми
случаями. Эти проверки **не заменяют общего доказательства**.

## Что представить

Сдайте этот ноутбук с ответами задания 1, полными решениями заданий 2 и 3 и,
по желанию, задания 4*: выводами формул, таблицами импульсов, графиком модуля
базис-вектора и обоснованием оптимальности.
Использование чекеров и сохранение результатов их вызова не требуются.

Перед отправкой перезапустите ядро, выполните ячейки решения по порядку и сохраните
ноутбук с результатами. Название файла: **`hw_4_name_surname.ipynb`**;
вместо `name_surname` укажите свои имя и фамилию латиницей.
